# Resampling of DCLP5

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

### Read Existing Tables

In [2]:
file_path = "../../data/out/DCLP5_Dataset_2022-01-20-5e0f3b16-c890-4ace-9e3b-531f3687cf53/"
file_map = {
    'cgm': 'DCLP5_cgm_history.csv.gz',
    'bolus': 'DCLP5_bolus_event_history.csv.gz',
    'basal': 'DCLP5_basal_event_history.csv.gz',
}

In [3]:
def get_df_from_file(file_path, file_name, parse_datetime=True, usecols=None, sep=',', encoding='utf-8'):
    df = pd.read_csv(file_path + file_name, sep=sep, usecols=usecols, encoding=encoding)
    if parse_datetime:
        df['datetime'] = pd.to_datetime(df['datetime'], unit='s')
        df = df.rename(columns={'datetime': 'date'})
    return df

In [4]:
def get_extended_df(original_df, value_column):
    """
    Get a df where quantities are distributed throughout 5-minute intervals instead of having start- and end dates.
    """
    new_rows = []
    for _, row in original_df.iterrows():
        new_rows.extend(split_duration(row, value_column))
    extended_df = pd.DataFrame(new_rows)
    extended_df.set_index('date', inplace=True)
    return extended_df

def split_duration(row, value_column):
    """
    For features with a duration, we split the values across 5-minute intervals by adding
    new rows for every 5-minute window in duration, and equally split the original quantity across those rows.
    """
    duration = row['end_date'] - row['date']
    rounded_duration = round(duration / pd.Timedelta(minutes=5)) * pd.Timedelta(minutes=5)
    num_intervals = rounded_duration // pd.Timedelta(minutes=5)
    if num_intervals < 1:
        num_intervals = 1
    value_per_interval = row[value_column] / num_intervals
    new_rows = []
    for i in range(int(num_intervals)):
        new_row = {
            'date': row['date'] + pd.Timedelta(minutes=5 * i),
            value_column: value_per_interval,
            'patient_id': row['patient_id'],
        }
        new_rows.append(new_row)
    return new_rows

In [5]:
df_glucose = get_df_from_file(file_path, file_map['cgm'])
df_bolus = get_df_from_file(file_path, file_map['bolus'])
df_basal = get_df_from_file(file_path, file_map['basal'])


### Resample Existing Tables

In [6]:
df_glucose.set_index('date', inplace=True)
df_glucose.head()

,patient_id,cgm
date,,
2019-04-17 12:07:54,26,107
2019-04-17 12:12:55,26,109
2019-04-17 12:17:53,26,125
2019-04-17 12:22:58,26,141
2019-04-17 12:32:57,26,140


In [7]:
df_bolus_orig = df_bolus.copy()
df_bolus['end_date'] = df_bolus['date'] + pd.to_timedelta(df_bolus['delivery_duration'], unit='s')
df_bolus = get_extended_df(df_bolus, 'bolus')
df_bolus

,bolus,patient_id
date,,
2018-08-02 14:36:32,0.530000,66
2019-05-02 18:04:08,1.302500,95
2018-12-19 13:54:57,1.078200,20
2019-02-07 20:14:46,0.360000,87
2019-07-25 09:58:36,0.628700,99
...,...,...
2019-12-17 20:28:58,0.048575,99
2020-01-11 13:45:59,0.303500,99
2020-01-11 13:50:59,0.303500,99


In [8]:
print(f'New sum after distribution of extended boluses: {df_bolus["bolus"].sum():.2f}, should be: {df_bolus_orig["bolus"].sum():.2f}')

New sum after distribution of extended boluses: 485249.65, should be: 485249.65


In [9]:
df_basal_orig = df_basal.copy()
df_basal.sort_values(by=['patient_id', 'date'], inplace=True)
df_basal.set_index('date', inplace=True)
df_basal

,patient_id,basal_rate
date,,
2018-06-16 18:11:54,1,0.000
2018-06-20 17:35:15,1,0.000
2018-06-23 02:04:50,1,0.000
2018-06-23 02:31:35,1,0.000
2018-07-04 22:21:30,1,0.000
...,...,...
2019-02-17 09:02:45,101,1.225
2019-02-17 09:07:45,101,0.450
2019-02-17 09:53:48,101,0.893


In [10]:
processed_dfs = []
subject_ids = df_glucose['patient_id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_glucose[df_glucose['patient_id'] == subject_id].copy()
    df_subject = df_subject[['cgm']].resample('5min', label='right').mean()
    df_subject['patient_id'] = subject_id
    df_subject.sort_index(inplace=True)

    def merge_data(df_col, df_subject, col_names, subject_id, agg_type='sum'):
        """ agg_type is data aggregation type. """
        df_subset = df_col[df_col['patient_id'] == subject_id].copy()
        if not df_subset.empty:
            if agg_type == 'mean':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').mean()
            elif agg_type == 'sum':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            elif agg_type == 'first':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').first()
            elif agg_type == 'last':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').last()
            else:
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            df_subject = pd.merge(df_subject, df_subset, on="date", how='outer')
        else:
            df_subject[col_names] = np.nan
        return df_subject

    # Add insulin and insulin type
    df_subject = merge_data(df_bolus, df_subject, ['bolus'], subject_id, agg_type='sum')
    df_subject = merge_data(df_basal, df_subject, ['basal_rate'], subject_id, agg_type='last')
    df_subject['basal_rate'] = df_subject['basal_rate'].ffill()
    
    df_subject['patient_id'] = subject_id
    df_subject = df_subject.rename(columns={'patient_id': 'id', 'basal_rate': 'basal', 'cgm': 'CGM'})
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)

Subjects: 100
26 is finished processing
83 is finished processing
60 is finished processing
63 is finished processing
3 is finished processing
93 is finished processing
42 is finished processing
101 is finished processing
6 is finished processing
53 is finished processing
73 is finished processing
1 is finished processing
66 is finished processing
10 is finished processing
22 is finished processing
97 is finished processing
96 is finished processing
38 is finished processing
39 is finished processing
40 is finished processing
47 is finished processing
25 is finished processing
80 is finished processing
87 is finished processing
32 is finished processing
35 is finished processing
49 is finished processing
62 is finished processing
55 is finished processing
84 is finished processing
7 is finished processing
82 is finished processing
21 is finished processing
61 is finished processing
20 is finished processing
88 is finished processing
76 is finished processing
92 is finished processing
7

In [11]:
df_final

,CGM,id,bolus,basal
date,,,,
2018-11-23 16:00:00,NaN,26,NaN,0.0
2018-11-23 16:05:00,NaN,26,NaN,0.0
2018-11-23 16:10:00,NaN,26,NaN,0.0
2018-11-23 16:15:00,NaN,26,NaN,0.0
2018-11-23 16:20:00,NaN,26,NaN,0.0
...,...,...,...,...
2019-08-09 12:55:00,163.0,65,NaN,1.1
2019-08-09 13:00:00,165.0,65,NaN,1.1
2019-08-09 13:05:00,153.0,65,NaN,1.1


### Add Additional Tables

We found:
- Carbs
- Insulin type
- Age
- Weight
- Height
- Gender

In [12]:
raw_data_file_path = "../../data/raw/DCLP5_Dataset_2022-01-20-5e0f3b16-c890-4ace-9e3b-531f3687cf53/"

In [13]:
# Adding carbs
df_carbs = get_df_from_file(raw_data_file_path, 'RocheMeter.txt', parse_datetime=False, sep='|')
#df_carbs['date'] = pd.to_datetime(df_carbs.DataDtTm_adjusted.fillna(df_carbs.DataDtTm))
#df_carbs = df_carbs[df_carbs['Carbs'] > 0][['PtID', 'date', 'Carbs']]
#df_carbs = df_carbs.rename(columns={'PtId': 'id', 'CarbInput': 'carbs'})
df_carbs.head()

,PtID,RecID,DataDtTm,BG,Carbs,IsQCTest,SystemDefinedEvents,UserDefinedEvents,Flags,IsQCtestJaeb
0,27,1,2018-11-10 08:38:00,300,NaN,True,Control,NaN,NaN,NaN
1,27,2,2018-11-10 08:37:29,44,NaN,True,Control,NaN,NaN,NaN
2,27,3,2018-11-05 19:34:32,524,NaN,False,NaN,NaN,NaN,NaN
3,27,4,2018-11-05 17:38:14,280,NaN,False,NaN,NaN,NaN,NaN
4,27,5,2018-11-05 14:52:32,191,NaN,False,NaN,NaN,NaN,NaN


In [14]:
# We observe that DCLP3 has no carb events
df_carbs[df_carbs['Carbs'].notna()]

,PtID,RecID,DataDtTm,BG,Carbs,IsQCTest,SystemDefinedEvents,UserDefinedEvents,Flags,IsQCtestJaeb


In [15]:
df_insulin_type = get_df_from_file(raw_data_file_path, 'Insulin.txt', parse_datetime=False, sep='|')
df_insulin_type = df_insulin_type[['PtID', 'ParentInsulinListID']]
df_insulin_type = df_insulin_type.rename(columns={'PtID': 'id', 'ParentInsulinListID': 'insulin_type'})
df_insulin_type

,id,insulin_type
0,21,Humalog (Lispro)
1,27,Humalog (Lispro)
2,26,Humalog (Lispro)
3,26,Humalog (Lispro)
4,80,Humalog (Lispro)
...,...,...
155,18,Novolog (Aspart)
156,95,Humalog (Lispro)
157,95,Novolog (Aspart)
158,12,Novolog (Aspart)


In [16]:
def add_single_value_to_subjects(df, df_new_val, col_name):
    # TODO: this function is very inefficient... add value directly to located rows instead
    processed_dfs = []
    subject_ids = df['id'].unique()
    for subject_id in subject_ids:
        df_subject = df[df['id'] == subject_id].copy()
        df_subject.sort_index(inplace=True)
    
        user_data = df_new_val[df_new_val['id'] == subject_id].copy()
        if not user_data.empty:
            df_subject[col_name] = user_data[col_name].iloc[0]
        else:
            df_subject[col_name] = np.nan        
        processed_dfs.append(df_subject)
        
    df = pd.concat(processed_dfs)
    return df

In [17]:
# Add insulin type
df_final = add_single_value_to_subjects(df_final, df_insulin_type, 'insulin_type')
df_final

,CGM,id,bolus,basal,insulin_type
date,,,,,
2018-11-23 16:00:00,NaN,26,NaN,0.0,Humalog (Lispro)
2018-11-23 16:05:00,NaN,26,NaN,0.0,Humalog (Lispro)
2018-11-23 16:10:00,NaN,26,NaN,0.0,Humalog (Lispro)
2018-11-23 16:15:00,NaN,26,NaN,0.0,Humalog (Lispro)
2018-11-23 16:20:00,NaN,26,NaN,0.0,Humalog (Lispro)
...,...,...,...,...,...
2019-08-09 12:55:00,163.0,65,NaN,1.1,Humalog (Lispro)
2019-08-09 13:00:00,165.0,65,NaN,1.1,Humalog (Lispro)
2019-08-09 13:05:00,153.0,65,NaN,1.1,Humalog (Lispro)


In [18]:
# Add age and gender
df_user_data = get_df_from_file(raw_data_file_path, 'DiabScreening.txt', parse_datetime=False, sep='|')
df_user_data = df_user_data[['PtID', 'Gender']]
df_user_data = df_user_data.rename(columns={'PtID': 'id', 'Gender': 'gender'})
df_user_data

,id,gender
0,21,M
1,27,M
2,26,F
3,80,M
4,87,F
...,...,...
96,51,M
97,18,M
98,95,F
99,12,F


In [19]:
# Add gender
df_final = add_single_value_to_subjects(df_final, df_user_data, 'gender')
df_final.head()

,CGM,id,bolus,basal,insulin_type,gender
date,,,,,,
2018-11-23 16:00:00,NaN,26,NaN,0.0,Humalog (Lispro),F
2018-11-23 16:05:00,NaN,26,NaN,0.0,Humalog (Lispro),F
2018-11-23 16:10:00,NaN,26,NaN,0.0,Humalog (Lispro),F
2018-11-23 16:15:00,NaN,26,NaN,0.0,Humalog (Lispro),F
2018-11-23 16:20:00,NaN,26,NaN,0.0,Humalog (Lispro),F


In [23]:
#df_weight_and_height = get_df_from_file(raw_data_file_path, 'DiabPhysExam.txt', parse_datetime=False, sep='|')
#df_weight_and_height = df_weight_and_height[['PtID', 'Weight', 'Height']]
#df_weight_and_height = df_weight_and_height.rename(columns={'PtID': 'id', 'Weight': 'weight', 'Height': 'height'})
#df_weight_and_height

# Load the data
df_weight_and_height = get_df_from_file(
    raw_data_file_path,
    'DiabPhysExam.txt',
    parse_datetime=False,
    sep='|',
)

# Select and rename relevant columns
df_weight_and_height = df_weight_and_height[['PtID', 'Weight', 'Height', 'WeightUnits', 'HeightUnits']]
df_weight_and_height = df_weight_and_height.rename(columns={'PtID': 'id', 'Weight': 'weight', 'Height': 'height'})

# Convert weight to lbs
def convert_weight_to_lbs(row):
    if pd.isna(row['WeightUnits']):
        print(row)
        return np.nan
    if row['WeightUnits'].lower() == 'kg':
        return row['weight'] * 2.20462
    return row['weight']

# Convert height to feet
def convert_height_to_feet(row):
    if pd.isna(row['HeightUnits']):
        return np.nan
    if row['HeightUnits'].lower() == 'cm':
        return row['height'] / 30.48
    elif row['HeightUnits'].lower() == 'in':
        return row['height'] / 12
    return row['height']  # assume already in feet if not cm/in

# Apply conversions
df_weight_and_height['weight'] = df_weight_and_height.apply(convert_weight_to_lbs, axis=1)
df_weight_and_height['height'] = df_weight_and_height.apply(convert_height_to_feet, axis=1)

# Optionally drop units columns if no longer needed
df_weight_and_height = df_weight_and_height.drop(columns=['WeightUnits', 'HeightUnits'])

df_weight_and_height

id                65
weight           NaN
height         154.0
WeightUnits      NaN
HeightUnits       cm
Name: 157, dtype: object


,id,weight,height
0,21,152.000000,5.685696
1,21,162.000000,5.675853
2,21,170.000000,5.725000
3,27,63.933980,4.232283
4,27,71.000000,4.330709
...,...,...,...
295,12,74.957080,4.593176
296,12,80.027706,4.730971
297,74,111.333310,5.413386
298,74,114.640240,5.413386


In [24]:
# Add weight and height
df_final = add_single_value_to_subjects(df_final, df_weight_and_height, 'weight')
df_final = add_single_value_to_subjects(df_final, df_weight_and_height, 'height')
df_final.head()

,CGM,id,bolus,basal,insulin_type,gender,weight,height
date,,,,,,,,
2018-11-23 16:00:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979
2018-11-23 16:05:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979
2018-11-23 16:10:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979
2018-11-23 16:15:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979
2018-11-23 16:20:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979


In [25]:
df_age = get_df_from_file(raw_data_file_path, 'PtRoster.txt', parse_datetime=False, sep='|')
df_age = df_age[['PtID', 'AgeAtEnrollment']]
df_age = df_age.rename(columns={'PtID': 'id', 'AgeAtEnrollment': 'age'})
df_age

,id,age
0,21,13
1,27,8
2,26,10
3,80,11
4,87,11
...,...,...
96,51,13
97,18,11
98,95,11
99,12,10


In [26]:
df_final = add_single_value_to_subjects(df_final, df_age, 'age')
df_final.head()


,CGM,id,bolus,basal,insulin_type,gender,weight,height,age
date,,,,,,,,,
2018-11-23 16:00:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979,10
2018-11-23 16:05:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979,10
2018-11-23 16:10:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979,10
2018-11-23 16:15:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979,10
2018-11-23 16:20:00,NaN,26,NaN,0.0,Humalog (Lispro),F,104.5,4.2979,10


In [27]:
# Basal U/hr to U
df_final['basal'] = df_final['basal'] / 12

### Save Resampled Data

In [28]:
df_final

,CGM,id,bolus,basal,insulin_type,gender,weight,height,age
date,,,,,,,,,
2018-11-23 16:00:00,NaN,26,NaN,0.000000,Humalog (Lispro),F,104.500000,4.297900,10
2018-11-23 16:05:00,NaN,26,NaN,0.000000,Humalog (Lispro),F,104.500000,4.297900,10
2018-11-23 16:10:00,NaN,26,NaN,0.000000,Humalog (Lispro),F,104.500000,4.297900,10
2018-11-23 16:15:00,NaN,26,NaN,0.000000,Humalog (Lispro),F,104.500000,4.297900,10
2018-11-23 16:20:00,NaN,26,NaN,0.000000,Humalog (Lispro),F,104.500000,4.297900,10
...,...,...,...,...,...,...,...,...,...
2019-08-09 12:55:00,163.0,65,NaN,0.091667,Humalog (Lispro),F,101.192058,5.036089,10
2019-08-09 13:00:00,165.0,65,NaN,0.091667,Humalog (Lispro),F,101.192058,5.036089,10
2019-08-09 13:05:00,153.0,65,NaN,0.091667,Humalog (Lispro),F,101.192058,5.036089,10


In [29]:
save_file_path = "../../data/resampled/"
os.makedirs(save_file_path, exist_ok=True)
df_final.to_csv(save_file_path + 'DCLP5.csv')